# Stop Guessing: A Smarter Way to Infer Time Frequencies in Climate Data

*A practical demonstration of robust frequency inference for climate time series.*

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import cftime

# old package is named pymor but now it is pycmor
from pymor.core.infer_freq import infer_frequency, FrequencyResult

## The Problem: Three Ways `xarray.infer_freq()` Returns `None`

`pandas` and `xarray` infer frequency by expecting perfectly regular, standard-calendar timestamps. Real climate model output rarely satisfies all three of those conditions at once.

### 1. Non-standard calendars

Climate models routinely use calendars that standard Python `datetime` cannot represent—`noleap` (365 days every year), `360_day` (12 × 30 days), and others. `xarray` delegates to `pandas` for inference, which doesn't understand `cftime` objects at all.

In [ ]:
times = [cftime.Datetime360Day(2000, m, 16) for m in range(1, 5)]

try:
    print(f"xarray result: {xr.infer_freq(times)}")
except Exception as e:
    print(f"xarray failed: {e}")

print(f"pycmor result: {infer_frequency(times)}")

xarray failed: <class 'cftime._cftime.Datetime360Day'> is not convertible to datetime, at position 0
pycmor result: M


### 2. Unanchored (shifted) timestamps

Monthly means are often stamped mid-month rather than on the first or last day. The spacing is still ~30 days, but `xarray.infer_freq` requires stamps to fall on a recognised anchor.

In [ ]:
# Monthly data stamped on the 6th of each month instead of the 1st
times = pd.date_range("2000-01-01", periods=4, freq="MS") + pd.Timedelta(days=5)
print("Timestamps:", list(times))
print(f"xarray result:  {xr.infer_freq(times)}")
print(f"pycmor result:  {infer_frequency(times)}")

Timestamps: [Timestamp('2000-01-06 00:00:00'), Timestamp('2000-02-06 00:00:00'), Timestamp('2000-03-06 00:00:00'), Timestamp('2000-04-06 00:00:00')]
xarray result:  None
pycmor result:  M


### 3. Missing steps or duplicates

A single missing month, or duplicate timestamps from accidentally concatenating the same file twice, is enough to make `xarray.infer_freq` return `None`.

In [ ]:
# March is missing — month-end stamps for Jan, Feb, Apr
times = pd.to_datetime(["2000-01-31", "2000-02-29", "2000-04-30"])
print(f"xarray result:  {xr.infer_freq(times)}")
print(f"pycmor result:  {infer_frequency(times)}")

xarray result:  None
pycmor result:  M


---

## The Fix: `pycmor.core.infer_freq`

The approach is straightforward:

- Compute **deltas between all consecutive time points**.
- Take the **median delta** — this smooths over gaps, duplicates, and small misalignments without rejecting the whole series.
- Convert all timestamps (including `cftime`) to a comparable numerical format before doing any arithmetic.

This makes the function resilient to the three failure modes above, across any calendar.

---

## Rich Diagnostics Instead of Silent Failure

Pass `return_metadata=True` to get a `FrequencyResult` object instead of a plain string. Instead of a silent `None`, you get a structured explanation of what was found and why it may be imperfect.

`strict=True` raises a `ValueError` instead of returning `None` when frequency cannot be determined — useful for pipeline assertions.

In [ ]:
times = ["2000-01-01", "2000-02-01", "2000-02-28", "2000-04-01"]

result = infer_frequency(times, return_metadata=True, strict=True)
print(f"frequency  : {result.frequency}")
print(f"delta_days : {result.delta_days}")
print(f"step       : {result.step}")
print(f"is_exact   : {result.is_exact}")
print(f"status     : {result.status}")

frequency  : M
delta_days : 27.0
step       : 1
is_exact   : False
status     : irregular


**What each field means:**

| Field | Description |
|-------|-------------|
| `frequency` | Inferred frequency string (`'D'`, `'M'`, `'MS'`, etc.) |
| `delta_days` | Median spacing between time steps, in days |
| `step` | Multiplier (e.g. `step=3` with `frequency='M'` means quarterly) |
| `is_exact` | `True` only if every spacing is identical |
| `status` | `'valid'`, `'missing_steps'`, `'irregular'`, or `'too_short'` |

**How to interpret `status`:**

- `valid` + `is_exact=True` → safe for resampling and analysis
- `missing_steps` → gaps present; consider filling before analysis
- `irregular` → underlying frequency detectable but spacing is inconsistent
- `too_short` → fewer than two points; cannot determine frequency

---

## End-to-End: Detecting Issues After File Concatenation

Combining NetCDF files from different sources is one of the most common sources of subtle time-axis corruption—overlapping chunks, a missing month, misaligned calendars. `infer_frequency` gives you a single call to catch all of it before it propagates into your analysis.

Here we simulate two files covering Jan–Jun and Jul–Dec 2000, where file 2 has a gap (July 15 is missing).

In [ ]:
# Simulate two NetCDF files
file1_times = pd.date_range("2000-01-01", "2000-06-30", freq="D")

file2_part1 = pd.date_range("2000-07-01", "2000-07-14", freq="D")
file2_part2 = pd.date_range("2000-07-16", "2000-12-31", freq="D")
file2_times = file2_part1.append(file2_part2)

# Check each file individually before combining
for i, times in enumerate([file1_times, file2_times], 1):
    result = infer_frequency(times, return_metadata=True, strict=True)
    print(f"File {i}: status={result.status!r}, is_exact={result.is_exact}")

File 1: status='valid', is_exact=True
File 2: status='missing_steps', is_exact=False


Catching problems per-file first is preferable to discovering them after combining hundreds of files. But `infer_frequency` works equally well on the combined time axis.

In [ ]:
# Use np.concatenate (not .union) to preserve duplicates and ordering
combined_times = pd.DatetimeIndex(
    np.concatenate([file1_times, file2_times])
)

result = infer_frequency(combined_times, return_metadata=True, strict=True)
print(f"Combined: status={result.status!r}, frequency={result.frequency!r}")

Combined: status='missing_steps', frequency='D'


The call costs microseconds and can be inserted as an assertion at any pipeline stage:

In [ ]:
def load_and_validate(ds):
    """Raise if the time axis has gaps or irregular spacing."""
    result = infer_frequency(ds.time, return_metadata=True, strict=True)
    if result.status != "valid" or not result.is_exact:
        raise ValueError(
            f"Time axis issue: status={result.status!r}, frequency={result.frequency!r}"
        )
    return ds

# Quick demo with a clean dataset
times = pd.date_range("2000-01-01", periods=12, freq="MS")
ds_clean = xr.Dataset(coords={"time": times})
validated = load_and_validate(ds_clean)
print("Clean dataset passed validation.")

Clean dataset passed validation.


---

## Takeaway

`pycmor.core.infer_freq` addresses the three concrete ways `xarray.infer_freq` silently fails on real climate data:

- non-standard calendars → handled via cftime-aware delta computation
- unanchored timestamps → handled via median-based inference
- gaps and duplicates → surfaced via `FrequencyResult.status`

The diagnostics turn a silent `None` into an actionable signal. Insert one call after loading or concatenating files and you have an early-warning system for the most common class of time-axis corruption.

---

## Project Repository

- GitHub: [esm-tools/pycmor](https://github.com/esm-tools/pycmor)

---

## Authors

This work was developed by the High Performance Computing and Data Processing
group at the Alfred Wegener Institute for Polar and Marine Research (AWI),
Bremerhaven, Germany.

- Pavan Kumar Siligam (AWI) — [ORCID: 0009-0003-8054-7021](https://orcid.org/0009-0003-8054-7021)
- Paul Gierz (AWI) — [ORCID: 0000-0002-4512-087X](https://orcid.org/0000-0002-4512-087X)
- Miguel Andrés-Martínez (AWI) — [ORCID: 0000-0002-1525-5546](https://orcid.org/0000-0002-1525-5546)